# Results Explained

This notebook loads the saved router result data and creates simple visualizations for comparing model groups, baselines, and router performance.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

ROOT = Path.cwd().resolve()
if (ROOT / "router_model_results_summary.csv").exists():
    RESULTS_DIR = ROOT
    PROJECT_ROOT = ROOT.parents[1]
elif (ROOT / "results" / "router_summary" / "router_model_results_summary.csv").exists():
    PROJECT_ROOT = ROOT
    RESULTS_DIR = ROOT / "results" / "router_summary"
elif (ROOT.parent / "results" / "router_summary" / "router_model_results_summary.csv").exists():
    PROJECT_ROOT = ROOT.parent
    RESULTS_DIR = ROOT.parent / "results" / "router_summary"
else:
    raise FileNotFoundError("Could not find results/router_summary/router_model_results_summary.csv")
summary_path = RESULTS_DIR / "router_model_results_summary.csv"

router_results_df = pd.read_csv(summary_path)
numeric_columns = ["test_mae", "test_mse", "test_rmse"]
router_results_df[numeric_columns] = router_results_df[numeric_columns].apply(pd.to_numeric)

print(f"Loaded {len(router_results_df)} rows from {summary_path}")
display(router_results_df.round({"test_mae": 6, "test_mse": 6, "test_rmse": 6}))

## Best Method Per Model Count

In [ ]:
best_by_count_df = (
    router_results_df.sort_values(["model_count", "test_mae"], ignore_index=True)
    .groupby("model_count", as_index=False)
    .first()
)

display(
    best_by_count_df[
        ["model_count", "selected_models", "method", "test_mae", "test_mse", "test_rmse"]
    ].round({"test_mae": 6, "test_mse": 6, "test_rmse": 6})
)

## Best MAE By Model Count

## Learned Router Results

The one-model case has no learned router because there is no expert choice to make.

In [ ]:
router_only_df = router_results_df[
    router_results_df["method"].eq("Learned prediction-aware router")
].copy()

display(
    router_only_df[
        ["model_count", "selected_models", "method", "test_mae", "test_mse", "test_rmse"]
    ].round({"test_mae": 6, "test_mse": 6, "test_rmse": 6})
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.bar(
    best_by_count_df["model_count"].astype(str),
    best_by_count_df["test_mae"],
    color="#2f6f73",
)
ax.set_title("Best Test MAE by Number of Experts")
ax.set_xlabel("Number of selected experts")
ax.set_ylabel("Test MAE, lower is better")
ax.grid(axis="y", alpha=0.25)
for bar, value in zip(bars, best_by_count_df["test_mae"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
output_path = RESULTS_DIR / "results_explained_best_mae_by_count.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight")
print(f"Saved chart to {output_path}")
plt.show()

## Router And Baseline Comparison

In [ ]:
comparison_methods = [
    "Learned prediction-aware router",
    "Fixed validation-based soft weights",
    "Fixed equal average",
    "Validation-selected best expert",
]
comparison_df = router_results_df[router_results_df["method"].isin(comparison_methods)].copy()
pivot_df = comparison_df.pivot_table(
    index="model_count",
    columns="method",
    values="test_mae",
    aggfunc="first",
).sort_index()
pivot_df = pivot_df[[method for method in comparison_methods if method in pivot_df.columns]]

display(pivot_df.round(6))

ax = pivot_df.plot(kind="bar", figsize=(11, 5), width=0.82)
ax.set_title("Router vs Baselines by Expert Count")
ax.set_xlabel("Number of selected experts")
ax.set_ylabel("Test MAE, lower is better")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
output_path = RESULTS_DIR / "results_explained_router_vs_baselines.png"
plt.savefig(output_path, dpi=160, bbox_inches="tight")
print(f"Saved chart to {output_path}")
plt.show()

## Oracle Combination Potential

## Router Significance Tests

These tests compare the learned router against each baseline on the same test windows. The improvement is `baseline MAE - router MAE`, so positive numbers mean the router is better.

In [ ]:
significance_path = RESULTS_DIR / "router_significance_tests.csv"
if significance_path.exists():
    significance_df = pd.read_csv(significance_path)
    display(
        significance_df[
            significance_df["strongest_baseline"].eq(True)
        ][
            [
                "model_group",
                "baseline_method",
                "router_mae",
                "baseline_mae",
                "mae_improvement_baseline_minus_router",
                "bootstrap_95_ci_low",
                "bootstrap_95_ci_high",
                "holm_adjusted_p_value",
                "significant_at_0_05",
                "n_windows",
            ]
        ].round(6)
    )
else:
    print("No significance file found. Run: python scripts/router_significance_tests.py --device cpu")

In [ ]:
if significance_path.exists():
    strongest_df = significance_df[significance_df["strongest_baseline"].eq(True)].copy()
    strongest_df = strongest_df.sort_values("model_count")
    y = range(len(strongest_df))
    x = strongest_df["mae_improvement_baseline_minus_router"]
    xerr = [
        x - strongest_df["bootstrap_95_ci_low"],
        strongest_df["bootstrap_95_ci_high"] - x,
    ]

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.errorbar(x, y, xerr=xerr, fmt="o", color="#2f6f73", capsize=4)
    ax.axvline(0, color="black", linewidth=1, alpha=0.7)
    ax.set_yticks(list(y))
    ax.set_yticklabels(strongest_df["model_group"])
    ax.set_xlabel("MAE improvement vs strongest baseline; positive means router is better")
    ax.set_title("Learned Router Improvement with 95% Bootstrap Confidence Intervals")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    output_path = RESULTS_DIR / "results_explained_router_significance.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    print(f"Saved chart to {output_path}")
    plt.show()

In [ ]:
oracle_path = RESULTS_DIR / "candidate_expert_combinations.csv"
if oracle_path.exists():
    oracle_df = pd.read_csv(oracle_path)
    oracle_df["oracle_mae"] = pd.to_numeric(oracle_df["oracle_mae"])
    best_oracle_by_size_df = (
        oracle_df.sort_values(["number_of_experts", "oracle_mae"], ignore_index=True)
        .groupby("number_of_experts", as_index=False)
        .first()
    )
    display(best_oracle_by_size_df[["number_of_experts", "combination", "oracle_mae"]].round({"oracle_mae": 6}))

    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.plot(
        best_oracle_by_size_df["number_of_experts"],
        best_oracle_by_size_df["oracle_mae"],
        marker="o",
        linewidth=2,
        color="#7a4f9f",
    )
    ax.set_title("Best Oracle MAE by Expert Count")
    ax.set_xlabel("Number of experts in oracle combination")
    ax.set_ylabel("Oracle MAE, lower is better")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    output_path = RESULTS_DIR / "results_explained_oracle_mae_by_count.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    print(f"Saved chart to {output_path}")
    plt.show()
else:
    print(f"No oracle combination file found at {oracle_path}")

## Extra Router Result Files

In [ ]:
ROUTER2_SUMMARY_DIR = PROJECT_ROOT / "results" / "router2_summary"
extra_patterns = [
    "routerdc_hard_test_comparison.csv",
    "router2_test_comparison.csv",
    "router2_*_test_comparison.csv",
]
extra_files = []
for pattern in extra_patterns:
    extra_files.extend(sorted(ROUTER2_SUMMARY_DIR.glob(pattern)))

seen = set()
extra_files = [path for path in extra_files if not (path in seen or seen.add(path))]

if not extra_files:
    print("No extra Router2 or RouterDC comparison files found yet.")
else:
    for path in extra_files:
        print(path.relative_to(PROJECT_ROOT))
        df = pd.read_csv(path)
        for column in ["Test MAE", "Test MSE", "Test RMSE"]:
            if column in df.columns:
                df[column] = pd.to_numeric(df[column])
        display(df.round(6))